# SOXX 2010–2022 原始披露集中度完整 Stage1 来源映射

预先冻结 48 个基准原始申报事件、3020 个原 Stage1 训练信号日、SEC 申报日后的实际 Qlib 交易日生效、后续修订未决遮罩。只做来源映射；不读 SOXX 收益、不拟合、不跑账户。

In [1]:
import os,json,hashlib,pickle,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from datetime import datetime
import pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxx_2010_2022_concentration_source_map_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for path,digest in plan['sources_sha256'].items():assert sha(root/path)==digest,('source drift',path)
group=plan['source_groups'];anchor=json.loads((root/group['pre_Stage1_anchor']).read_text());anchor_summary=json.loads((root/group['anchor_summary']).read_text());series=json.loads((root/group['series_calendar_and_amendments']).read_text());same_amendment=json.loads((root/group['2016_Sep_amendment_same_equity']).read_text());bridge_summary=json.loads((root/group['2018_Sep_to_2022_bridge']).read_text())
assert anchor['status']=='single_original_SOXX_reported_equity_concentration_parser_passed_source_only' and anchor['parsed_equity_rows']==30 and anchor['total_closure_difference_usd']==0
assert same_amendment['status']=='amendment_SOXX_equity_schedule_numeric_qualified_unchanged'
assert series['base_filing_rows']==32 and series['amendment_rows']==5 and bridge_summary['report_events']==16 and bridge_summary['nas_readback']['verified']
state={'status':'started','new_GETs':0,'source_numeric_result_reads':0,'Qlib_source_Recorder_object_loads':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
base_index={x['accession']:x for x in series['rows'] if '/A' not in x['form']};assert len(base_index)==32
old=[]
for path in group['2011_2018_base_numeric_results']:
 result=json.loads((root/path).read_text());state['source_numeric_result_reads']+=1
 assert result['reports_attempted']==result['reports_with_numeric_closure']==len(result['reports'])
 for row in result['reports']:
  accession=row['accession'];meta=base_index[accession];assert row['report_date']==meta['report_date'] and row['parsed_equity_rows']==30 and row['total_closure_difference_usd']==0 and all(row['checks'].values())
  old.append({'accession':accession,'report_period':row['report_date'],'filed':meta['filed'],'accepted_at_utc':row['accepted_at_utc'],'form':row['form'],'top5':row['top5_disclosed_equity_value_share'],'HHI':row['herfindahl_sum_squared_disclosed_equity_weights'],'source_sha256':row['raw_sha256'],'source_result_sha256':plan['sources_sha256'][path]})
assert len(old)==31 and len({x['accession'] for x in old})==31
state['source_numeric_result_reads']+=2;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
anchor_event={'accession':anchor_summary['pre_signal_candidate_accession'],'report_period':anchor['report_date'],'filed':anchor['accepted_at_utc'][:10],'accepted_at_utc':anchor['accepted_at_utc'],'form':'N-Q','top5':anchor['top5_disclosed_equity_value_share'],'HHI':anchor['herfindahl_sum_squared_disclosed_equity_weights'],'source_sha256':anchor['source_raw_sha256'],'source_result_sha256':plan['sources_sha256'][group['pre_Stage1_anchor']]}
assert anchor_event['filed']=='2010-12-30' and anchor_event['accession']=='0001193125-10-291216'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
with R.uri_context(bridge_summary['recorder_uri']):
 recorder=R.get_recorder(recorder_id=bridge_summary['recorder_id'],experiment_id=bridge_summary['experiment_id']);late=recorder.load_object('analysis/source_events')
state['Qlib_source_Recorder_object_loads']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
assert len(late)==16
late=late[['accession','report_period','filed','form','top5','HHI','source_sha256']].copy();late['accepted_at_utc']=None;late['source_result_sha256']=plan['sources_sha256'][group['2018_Sep_to_2022_bridge']]
events=pd.concat([pd.DataFrame([anchor_event]+old),late],ignore_index=True).sort_values('filed').reset_index(drop=True)
assert len(events)==48 and events.accession.is_unique and events.report_period.is_unique
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2023-01-10',freq='day'))
events['filed_date']=pd.to_datetime(events.filed)
events['effective_signal_date']=[calendar[int(calendar.searchsorted(d,side='right'))] for d in events.filed_date]
assert events.effective_signal_date.is_unique and (events.effective_signal_date>events.filed_date).all()
signals=calendar[(calendar>=pd.Timestamp(plan['train_span'][0]))&(calendar<=pd.Timestamp(plan['train_span'][1]))]
assert len(signals)==3020
mapped=pd.merge_asof(pd.DataFrame({'signal_date':signals}),events[['effective_signal_date','accession','report_period','filed_date','form','top5','HHI']],left_on='signal_date',right_on='effective_signal_date',direction='backward')
mapped['source_known']=mapped.accession.notna();mapped['source_age_days']=(mapped.signal_date-mapped.filed_date).dt.days
assert len(mapped)==3020 and mapped.signal_date.is_unique and mapped.source_known.all() and mapped.source_age_days.ge(1).all()
amendments=[]
for item in series['rows']:
 if '/A' not in item['form']:continue
 d=pd.Timestamp(item['filed']);effective=calendar[int(calendar.searchsorted(d,side='right'))]
 amendments.append({'accession':item['accession'],'form':item['form'],'report_period':item['report_date'],'filed':item['filed'],'accepted_at_utc':item['accepted_at_utc'],'effective_signal_date':effective,'equity_semantics_qualified':item['accession']=='0001193125-17-257378'})
assert len(amendments)==5 and sum(x['equity_semantics_qualified'] for x in amendments)==1
mapped['amendment_uncertain']=False
for amendment in amendments:
 if amendment['equity_semantics_qualified']:continue
 mask=(mapped.report_period==amendment['report_period'])&(mapped.signal_date>=amendment['effective_signal_date'])
 mapped.loc[mask,'amendment_uncertain']=True
mapped['qualified_source']=mapped.source_known&~mapped.amendment_uncertain
frame=mapped.set_index('signal_date')[['top5','HHI','source_known','amendment_uncertain','qualified_source','source_age_days']]
frame.index=pd.MultiIndex.from_arrays([signals,['SOXL_SOXX_CASH']*len(signals)],names=['datetime','instrument'])
static=pd.concat({'feature':frame[['top5','HHI']],'meta':frame[['source_known','amendment_uncertain','qualified_source','source_age_days']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'Stage1_train':tuple(plan['train_span'])})
prepared=dataset.prepare('Stage1_train',col_set=['feature','meta']);assert len(prepared)==3020 and int(prepared[('meta','source_known')].sum())==3020
uncertain=mapped[mapped.amendment_uncertain]
audit={'status':'full_Stage1_first_original_concentration_source_map_amendment_gap_explicit','source_plan_sha256':sha(card/'plan.json'),'base_report_events':len(events),'old_base_reports':len(old),'late_bridge_events':len(late),'Stage1_Qlib_train_sessions':len(signals),'first_version_source_known_sessions':int(mapped.source_known.sum()),'qualified_after_amendment_mask_sessions':int(mapped.qualified_source.sum()),'amendment_uncertain_sessions':len(uncertain),'amendment_uncertain_first_date':str(uncertain.signal_date.min().date()) if len(uncertain) else None,'amendment_uncertain_last_date':str(uncertain.signal_date.max().date()) if len(uncertain) else None,'amendments_total':len(amendments),'amendment_equity_semantics_qualified':sum(x['equity_semantics_qualified'] for x in amendments),'first_source_effective_signal':str(events.effective_signal_date.min().date()),'last_effective_signal':str(events.effective_signal_date.max().date()),'max_source_age_days':int(mapped.source_age_days.max()),'top5_source_bounds':[float(events.top5.min()),float(events.top5.max())],'HHI_source_bounds':[float(events.HHI.min()),float(events.HHI.max())],'full_Stage1_concentration_source_qualified':bool(mapped.qualified_source.all()),'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True);assert len(pickle.loads(pickle.dumps(dataset)).prepare('Stage1_train',col_set=['feature','meta']))==3020
tracking=card/'qlib_recorder'
with R.start(experiment_name='SOXX_original_concentration_full_Stage1_source_map',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes=plan['sources_sha256'])
 rec.save_objects(artifact_path='analysis',source_events=events,amendments=amendments,mapped_signals=mapped,source_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_report_events=len(events),Stage1_train_sessions=len(signals),source_known_sessions=int(mapped.source_known.sum()),amendment_uncertain_sessions=len(uncertain),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id);assert len(checked.load_object('analysis/source_events'))==48 and len(checked.load_object('metadata/dataset').prepare('Stage1_train',col_set=['feature','meta']))==3020
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'SOXX_original_concentration_full_Stage1_source_map','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-full-source-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':card.name,'decision':'first_original_Stage1_concentration_calendar_complete_2017_amendment_gap_blocks_full_source_qualification','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','base_report_events':48,'model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:audit[k] for k in ('base_report_events','Stage1_Qlib_train_sessions','first_version_source_known_sessions','qualified_after_amendment_mask_sessions','amendment_uncertain_sessions','amendment_uncertain_first_date','amendment_uncertain_last_date','full_Stage1_concentration_source_qualified')});print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'base_report_events': 48, 'Stage1_Qlib_train_sessions': 3020, 'first_version_source_known_sessions': 3020, 'qualified_after_amendment_mask_sessions': 3009, 'amendment_uncertain_sessions': 11, 'amendment_uncertain_first_date': '2017-08-15', 'amendment_uncertain_last_date': '2017-08-29', 'full_Stage1_concentration_source_qualified': False}
{'recorder_id': 'd128dfd4beca4cf384e428e45a14707f', 'nas_sha256': 'fecdd156e648eb8121e2fe09bd35667113792bd5025f4278fc507938e51f0c0d'}


## 中文结论

48 个原始 SOXX 普通股披露事件按 SEC 申报日之后的真实 Qlib 交易日映射，冻结 Stage1 2011–2022 年 3,020 个训练日均有首版来源。2017-08-15 至 2017-08-29 的 11 日因 2017-03 报告迟到修订尚未核对而明确未决；合格来源 3,009/3,020，不能拟合。DatasetH/Recorder `d128dfd4beca4cf384e428e45a14707f` 与 NAS SHA-256 `fecdd156e648eb8121e2fe09bd35667113792bd5025f4278fc507938e51f0c0d` 回读通过。0 收益读取、0 拟合、0 账户。下一步仅核对影响 11 日的确切修订。